# Notebook 2: Exploratory Data Analysis (EDA) 


The process conducted in this notebook involves the application of an Exploratory Data Analysis on the ‘candidates’ data migrated to the MySQL database.Basically, Exploratory Data Analysis (EDA) is the process of visually and statistically summarizing, exploring, and understanding the main characteristics, patterns, and relationships within a dataset.

In [1]:
import sys
import os
import numpy as np 
from dotenv import load_dotenv
import psycopg2 
from sqlalchemy import create_engine, text, types as sqltypes, Column
from sqlalchemy import Integer, SmallInteger, Float, Numeric, DateTime, Boolean, String, Time
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import declarative_base 
import pandas as pd

In [2]:
# Add the 'src' directory to the PYTHONPATH
sys.path.append(os.path.abspath('../src'))

# Importing the utility function from the db_utils module within the connection package
from connection.db_utils import get_db_connection

# Get the database connection
connection = get_db_connection()

Connected to the database successfully


In [3]:
df = pd.read_sql_table("candidates", connection)

df.head(3)

,id,first_name,last_name,email,application_date,country,yoe,seniority,technology,code_challenge_score,technical_interview_score
0,1,Bernadette,Langworth,leonard91@yahoo.com,26/02/2021,Norway,2,Intern,Data Engineer,3,3
1,2,Camryn,Reynolds,zelda56@hotmail.com,09/09/2021,Panama,10,Intern,Data Engineer,2,10
2,3,Larue,Spinka,okey_schultz41@gmail.com,14/04/2020,Belarus,4,Mid-Level,Client Success,10,9


In [4]:
# Display initial data types
print("Before Conversion:")
print(df.info())

Before Conversion:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   id                         50000 non-null  int64 
 1   first_name                 50000 non-null  object
 2   last_name                  50000 non-null  object
 3   email                      50000 non-null  object
 4   application_date           50000 non-null  object
 5   country                    50000 non-null  object
 6   yoe                        50000 non-null  int64 
 7   seniority                  50000 non-null  object
 8   technology                 50000 non-null  object
 9   code_challenge_score       50000 non-null  int64 
 10  technical_interview_score  50000 non-null  int64 
dtypes: int64(4), object(7)
memory usage: 4.2+ MB
None


In [5]:
# Use .values to get 'application_date' column as a list
application_date_values = df['application_date'].values.tolist()

# Show the list
print("List of 'application_date' Column")
print(application_date_values)

List of 'application_date' Column
['26/02/2021', '09/09/2021', '14/04/2020', '01/10/2020', '20/05/2020', '17/08/2019', '18/05/2018', '09/12/2021', '13/03/2018', '08/04/2022', '22/09/2019', '15/07/2020', '27/12/2021', '09/05/2020', '12/10/2019', '18/10/2018', '25/03/2020', '23/05/2021', '02/05/2018', '13/04/2021', '21/03/2019', '12/07/2020', '09/01/2021', '27/07/2021', '18/11/2019', '25/04/2018', '26/01/2020', '04/12/2021', '05/01/2022', '28/11/2020', '25/05/2019', '21/09/2018', '04/03/2018', '19/08/2021', '21/08/2018', '24/06/2020', '13/03/2018', '07/01/2019', '26/01/2022', '16/12/2019', '25/03/2022', '04/10/2018', '02/12/2019', '10/09/2020', '07/03/2021', '05/07/2021', '09/04/2021', '26/04/2019', '24/12/2020', '28/03/2021', '23/09/2020', '05/03/2018', '26/06/2021', '27/02/2022', '26/03/2022', '26/10/2018', '06/09/2021', '01/10/2021', '11/06/2018', '01/01/2021', '04/06/2019', '28/08/2018', '30/04/2018', '13/01/2022', '24/01/2021', '25/05/2018', '26/01/2020', '18/07/2021', '11/04/2022',

In [6]:
# Converting 'application_date' column to datetime

try:
        df['application_date'] = pd.to_datetime(df['application_date'], dayfirst=True)
        print(df['application_date'].iloc[:3])
except (ValueError, TypeError) as e:
        print(f"Error processing date column '{application_date}': {e}")


# Display data types after conversion
print("\nAfter Conversion:")
print(df.info())

0   2021-02-26
1   2021-09-09
2   2020-04-14
Name: application_date, dtype: datetime64[ns]

After Conversion:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   id                         50000 non-null  int64         
 1   first_name                 50000 non-null  object        
 2   last_name                  50000 non-null  object        
 3   email                      50000 non-null  object        
 4   application_date           50000 non-null  datetime64[ns]
 5   country                    50000 non-null  object        
 6   yoe                        50000 non-null  int64         
 7   seniority                  50000 non-null  object        
 8   technology                 50000 non-null  object        
 9   code_challenge_score       50000 non-null  int64         
 10  technical_interview_

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
# Find rows with duplicate emails
duplicate_emails = df[df.duplicated(subset=['email'], keep=False)]
duplicate_emails["email"].value_counts()


email
marianne31@yahoo.com        3
fern70@gmail.com            3
sandra83@gmail.com          2
dewayne50@gmail.com         2
matilda17@gmail.com         2
                           ..
marjolaine91@hotmail.com    2
jazmin54@gmail.com          2
reyna2@hotmail.com          2
kasandra68@hotmail.com      2
easter75@gmail.com          2
Name: count, Length: 165, dtype: int64

In [13]:
print(f"Found {len(duplicate_emails)} rows with duplicate emails.")

Found 332 rows with duplicate emails.


In [9]:
def process_candidate_uniqueness(df):
    """
    Checks for unique candidates (first_name, last_name, email)
    and then checks if those unique candidates have different application dates.
    Prints the number of rows dropped.
    """

    df['unique_candidate'] = True  # Initialize uniqueness flag
    df['different_application_dates'] = False #initialize different dates flag

    # Create candidate key
    df['candidate_key'] = df['first_name'] + '_' + df['last_name']  + '_' + df['email']

    # Find duplicate candidate keys
    duplicate_keys = df[df.duplicated(subset=['candidate_key'], keep=False)]

    # Mark duplicates as False
    df.loc[duplicate_keys.index, 'unique_candidate'] = False

    # Check for different application dates among unique candidates
    unique_candidates = df[df['unique_candidate']]  # Only process unique candidates

    for key, group in unique_candidates.groupby('candidate_key'):
        if len(group['application_date'].unique()) > 1:
            df.loc[group.index, 'different_application_dates'] = True

    #count how many rows will be dropped.
    original_row_count = len(df)
    df_dropped = df[df['unique_candidate'] == False]
    rows_dropped = len(df_dropped)

    #drop the rows.
    df = df[df['unique_candidate'] == True]

    #print the number of rows dropped.
    print(f"Number of rows dropped: {rows_dropped}")

    df.drop('candidate_key', axis=1, inplace=True)
    return df


df = process_candidate_uniqueness(df)

Number of rows dropped: 0


In [10]:
# Check if the DataFrame is ordered by date
is_ordered = df['application_date'].is_monotonic_increasing

print(is_ordered)

False


In [12]:
# Sort the DataFrame by the 'date' column
df_sorted = df.sort_values(by='application_date')

In [14]:
min_values = df.min()
print(min_values)


id                                                          1
first_name                                            Aaliyah
last_name                                              Abbott
email                          aaliyah.bernhard55@hotmail.com
application_date                          2018-01-01 00:00:00
country                                           Afghanistan
yoe                                                         0
seniority                                           Architect
technology                           Adobe Experience Manager
code_challenge_score                                        0
technical_interview_score                                   0
unique_candidate                                         True
different_application_dates                             False
dtype: object


In [13]:
max_values = df.max()
print(max_values)


id                                                 50000
first_name                                          Zula
last_name                                         Zulauf
email                          zula_weissnat@hotmail.com
application_date                     2022-07-04 00:00:00
country                                         Zimbabwe
yoe                                                   30
seniority                                        Trainee
technology                             Technical Writing
code_challenge_score                                  10
technical_interview_score                             10
unique_candidate                                    True
different_application_dates                        False
dtype: object


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 13 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   id                           50000 non-null  int64         
 1   first_name                   50000 non-null  object        
 2   last_name                    50000 non-null  object        
 3   email                        50000 non-null  object        
 4   application_date             50000 non-null  datetime64[ns]
 5   country                      50000 non-null  object        
 6   yoe                          50000 non-null  int64         
 7   seniority                    50000 non-null  object        
 8   technology                   50000 non-null  object        
 9   code_challenge_score         50000 non-null  int64         
 10  technical_interview_score    50000 non-null  int64         
 11  unique_candidate             50000 non-nu